# Comprehensive Build Trace Analysis Guide

This notebook demonstrates how to analyze C++ build performance using Clang's `-ftime-trace` feature. We'll explore the trace analysis library and show practical techniques for understanding and improving compilation times.

## The Problem: C++ Metaprogramming Build Times

The Composable Kernel (CK) library uses extensive C++17 metaprogramming to generate high-performance GPU kernels. While this approach provides excellent runtime performance, it comes with a cost: **long compilation times**.

Understanding where the compiler spends its time is critical for:

- **Identifying bottlenecks**: Which templates are most expensive?
- **Measuring progress**: Are our optimizations working?
- **Focusing efforts**: Where should we invest time to improve build performance?

## The Solution: Data-Driven Analysis

Clang's `-ftime-trace` flag generates detailed JSON files showing exactly where compilation time is spent. This notebook shows how to:

1. **Parse** trace files efficiently using parallel processing
2. **Transform** raw JSON into structured multi-table schemas
3. **Analyze** build performance with relational queries
4. **Visualize** build parallelism with Gantt charts
5. **Identify** optimization opportunities

Let's treat this as a **big data problem** and use the best tools available: pandas, parallel processing, and Jupyter notebooks.

## Setup and Imports

In [ ]:
from importlib.util import find_spec
from multiprocessing import cpu_count
import pandas as pd
from pathlib import Path
import sys
import time

# Add parent directory to path to import trace_analysis
sys.path.insert(0, str(Path.cwd().parent))

from trace_analysis import TraceFile, TraceParser, TraceTransformer, find_trace_files

# Check for optional dependencies
HAS_TQDM = find_spec("tqdm") is not None
if not HAS_TQDM:
    print("Note: Install tqdm for progress bars: pip install tqdm")

HAS_PLOTLY = find_spec("plotly") is not None
if not HAS_PLOTLY:
    print("Note: Install plotly for visualizations: pip install plotly")

# Display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 80)

print(f"Using {cpu_count()} CPU cores for parallel processing")
print(f"Pandas version: {pd.__version__}")

## Part 1: Single File Analysis

Let's start by analyzing a single trace file to understand the schema structure.

We first identify all the json files from a build with `-ftime-trace` set in the `CXX_FLAGS`.

In [ ]:
# Configure the path to your trace files
TRACE_DIR = Path("../../../build-trace")

sample_files = find_trace_files(TRACE_DIR)

if not sample_files:
    print(f"No trace files found in {TRACE_DIR}")
    print("\nTo generate trace files:")
    print("1. Configure your build with: cmake -DCMAKE_CXX_FLAGS='-ftime-trace' ...")
    print("2. Build your project")
    print("3. Trace files will be generated alongside object files")
else:
    print(f"Found {len(sample_files):,} trace files")
    sample_file = sample_files[0]
    print(f"\nUsing sample file: {sample_file.name}")
    print(f"File size: {sample_file.stat().st_size / 1024:.1f} KB")

### Parsing to Pandas Data Frames

The Pandas schema creates separate DataFrames for:
- **templates**: Unique template definitions with structure
- **instantiations**: Template instantiation events with timing
- **template_args**: Template argument relationships
- **events**: All compiler events

This normalized structure enables efficient queries and reduces memory usage.

In [ ]:
if sample_files:
    # Parse the trace file
    trace_file = TraceFile.from_path(sample_file)

    start = time.time()
    events = TraceParser.parse(trace_file)
    parse_time = time.time() - start

    # Get beginning of time for timeline analysis
    import orjson

    with open(sample_file, "rb") as f:
        trace_data = orjson.loads(f.read())
    beginning_of_time = TraceTransformer.extract_beginning_of_time(trace_data)

    # Convert to enhanced schema
    start = time.time()
    tables = TraceTransformer.to_enhanced_schema(
        events, file_id=0, beginning_of_time_us=beginning_of_time
    )
    transform_time = time.time() - start

    print(f"Parsed {len(events):,} events in {parse_time:.3f}s")
    print(f"Transformed to Pandas tables in {transform_time:.3f}s\n")

    print("Pandas DataFrames:")
    for name, df in tables.items():
        mem_mb = df.memory_usage(deep=True).sum() / 1024**2
        cols = ", ".join(df.columns)
        print(f"  {name:20s}: {len(df):6,} rows, {mem_mb:6.2f} MB | {cols}")

Summarize the compilation unit (start time, duration, and any other summary statistics)

In [ ]:
if sample_files:
    print("Compilation Unit Summary:")
    print(f"  Trace file: {sample_file.name}")
    print(f"  Trace file size: {sample_file.stat().st_size / 1024:.1f} KB")
    print(f"  Start time: {pd.to_datetime(beginning_of_time, unit='us')}")
    print(f"  Total compilation time: {tables['events']['dur'].sum() / 1e6:.2f}s")
    print(f"  Total events: {len(tables['events']):,}")

### Examining the Templates Table

The templates table contains unique template definitions with parsed structure.

In [ ]:
if sample_files:
    templates_df = tables["templates"]

    # Filter to CK templates only (exclude std library templates)
    ck_templates_df = templates_df[
        templates_df["template_name"].str.startswith("ck::", na=False)
        | templates_df["template_name"].str.startswith("ck_tile::", na=False)
    ]

    print("Templates DataFrame Schema:")

    # Combine dtype and memory info
    mem_usage = templates_df.memory_usage(deep=True)
    total_mb = mem_usage.sum() / 1024**2

    print(f"{'Column':<25s} {'Type':<15s} {'Memory (MB)':>12s} {'% of Total':>12s}")
    print("-" * 67)
    for col in templates_df.columns:
        dtype_str = str(templates_df[col].dtype)
        mem_mb = mem_usage[col] / 1024**2
        pct = 100 * mem_usage[col] / mem_usage.sum()
        print(f"{col:<25s} {dtype_str:<15s} {mem_mb:12.2f} {pct:11.1f}%")

    # Add Index row
    idx_mem_mb = mem_usage["Index"] / 1024**2
    idx_pct = 100 * mem_usage["Index"] / mem_usage.sum()
    print(f"{'Index':<25s} {'RangeIndex':<15s} {idx_mem_mb:12.2f} {idx_pct:11.1f}%")
    print("-" * 67)
    print(f"{'TOTAL':<25s} {'':<15s} {total_mb:12.2f} {100.0:11.1f}%")

    print(f"\nTotal templates: {len(templates_df):,}")
    print(
        f"CK templates: {len(ck_templates_df):,} ({100 * len(ck_templates_df) / len(templates_df):.1f}%)"
    )
    print(f"Other templates: {len(templates_df) - len(ck_templates_df):,}")
    print("\nSample CK templates:")
    display(ck_templates_df.head(10))

### Template Instantiation Analysis

Join templates with instantiations to analyze performance.

In [ ]:
if sample_files and len(tables["instantiations"]) > 0:
    # Join instantiations with templates
    inst_df = tables["instantiations"].merge(
        tables["templates"][
            ["template_id", "template_name", "full_signature", "depth"]
        ],
        on="template_id",
    )

    total_template_time_us = inst_df["dur_us"].sum()
    total_time_us = tables["events"]["dur"].sum()

    print("Template Instantiation Summary:")
    print(f"  Unique templates: {len(tables['templates']):,}")
    print(f"  Total instantiations: {len(inst_df):,}")
    print(f"  Template time: {total_template_time_us / 1e6:.2f}s")
    print(f"  Percentage of build: {100 * total_template_time_us / total_time_us:.1f}%")
    print(f"  Avg per instantiation: {inst_df['dur_us'].mean() / 1e3:.2f} ms")

    # Most expensive templates by total time
    print("\nTop 10 Templates by Total Time:")
    template_stats = (
        inst_df.groupby("template_id")
        .agg(
            {
                "dur_us": ["count", "sum", "mean", "max"],
                "full_signature": "first",
                "depth": "first",
            }
        )
        .reset_index()
    )
    template_stats.columns = [
        "template_id",
        "count",
        "total_us",
        "mean_us",
        "max_us",
        "signature",
        "depth",
    ]
    template_stats["total_ms"] = template_stats["total_us"] / 1e3
    template_stats["mean_ms"] = template_stats["mean_us"] / 1e3

    display(
        template_stats.nlargest(10, "total_us")[
            ["signature", "count", "total_ms", "mean_ms", "depth"]
        ]
    )

### Template Depth Analysis

Analyze template nesting depth to identify complex metaprogramming patterns.

In [ ]:
if sample_files and len(tables["templates"]) > 0:
    depth_stats = (
        inst_df.groupby("depth").agg({"dur_us": ["count", "sum", "mean"]}).reset_index()
    )
    depth_stats.columns = ["depth", "count", "total_us", "mean_us"]
    depth_stats["total_ms"] = depth_stats["total_us"] / 1e3
    depth_stats["mean_ms"] = depth_stats["mean_us"] / 1e3

    print("Template Instantiation by Nesting Depth:")
    display(depth_stats[["depth", "count", "total_ms", "mean_ms"]])

## Part 2: Multi-File Analysis


⚠️ **Warning** The timings looks suspcious on some of these joined tables. We need to do more testing and development.


Now let's scale up to analyze an entire build using parallel processing.

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed


def process_file(json_path: Path, file_id: int) -> dict:
    """Process a single trace file and return enhanced schema tables."""
    import orjson
    from trace_analysis import TraceFile, TraceParser, TraceTransformer

    trace_file = TraceFile.from_path(json_path)
    events = TraceParser.parse(trace_file)

    # Extract beginning of time
    with open(json_path, "rb") as f:
        trace_data = orjson.loads(f.read())
    beginning_of_time = TraceTransformer.extract_beginning_of_time(trace_data)

    # Convert to Pandas DataFrames
    tables = TraceTransformer.to_enhanced_schema(
        events, file_id=file_id, beginning_of_time_us=beginning_of_time
    )

    return {"file_id": file_id, "file_name": json_path.name, "tables": tables}


print("Parallel processing function defined")

In [ ]:
# Find all trace files
json_files = find_trace_files(TRACE_DIR)

if not json_files:
    print(f"No trace files found in {TRACE_DIR}")
else:
    print(f"Found {len(json_files):,} trace files")
    total_size = sum(f.stat().st_size for f in json_files)
    print(f"Total size: {total_size / 1024**3:.2f} GB")

    # For demonstration, you might want to limit the number of files
    # Uncomment the next line to process only the first 100 files
    # json_files = json_files[:100]

### Processing All Files in Parallel

In [ ]:
if json_files:
    print(f"Processing {len(json_files):,} files with {cpu_count()} workers...\n")

    start_time = time.time()
    results = []

    # Submit all files for parallel processing
    with ProcessPoolExecutor(max_workers=cpu_count()) as executor:
        futures = {
            executor.submit(process_file, f, i): (f, i)
            for i, f in enumerate(json_files)
        }

        # Collect results with progress bar
        if HAS_TQDM:
            from tqdm.auto import tqdm

            pbar = tqdm(total=len(json_files), desc="Processing", unit="files")

        for future in as_completed(futures):
            result = future.result()
            results.append(result)

            if HAS_TQDM:
                pbar.update(1)

        if HAS_TQDM:
            pbar.close()

    parse_time = time.time() - start_time
    print(
        f"\nParsing complete in {parse_time:.2f}s ({len(json_files) / parse_time:.1f} files/sec)"
    )

    # Combine all tables
    print("\nCombining results...")
    combine_start = time.time()

    all_templates = []
    all_instantiations = []
    all_template_args = []
    all_events = []

    for result in results:
        tables = result["tables"]
        if len(tables["templates"]) > 0:
            all_templates.append(tables["templates"])
        if len(tables["instantiations"]) > 0:
            all_instantiations.append(tables["instantiations"])
        if len(tables["template_args"]) > 0:
            all_template_args.append(tables["template_args"])
        if len(tables["events"]) > 0:
            all_events.append(tables["events"])

    # Concatenate DataFrames
    templates_df = (
        pd.concat(all_templates, ignore_index=True) if all_templates else pd.DataFrame()
    )
    instantiations_df = (
        pd.concat(all_instantiations, ignore_index=True)
        if all_instantiations
        else pd.DataFrame()
    )
    template_args_df = (
        pd.concat(all_template_args, ignore_index=True)
        if all_template_args
        else pd.DataFrame()
    )
    events_df = (
        pd.concat(all_events, ignore_index=True) if all_events else pd.DataFrame()
    )

    combine_time = time.time() - combine_start
    total_time = time.time() - start_time

    print(f"Combined in {combine_time:.2f}s")
    print(f"\nTotal analysis time: {total_time:.2f}s")

    # Calculate memory usage
    total_memory = (
        templates_df.memory_usage(deep=True).sum()
        + instantiations_df.memory_usage(deep=True).sum()
        + template_args_df.memory_usage(deep=True).sum()
        + events_df.memory_usage(deep=True).sum()
    ) / 1024**3

    print("\nCombined Tables:")
    print(f"  Templates: {len(templates_df):,} rows")
    print(f"  Instantiations: {len(instantiations_df):,} rows")
    print(f"  Template Args: {len(template_args_df):,} rows")
    print(f"  Events: {len(events_df):,} rows")
    print(f"  Total memory: {total_memory:.2f} GB")

### Build-Wide Statistics

In [ ]:
if json_files and len(events_df) > 0:
    total_build_time_us = events_df["dur"].sum()
    total_template_time_us = instantiations_df["dur_us"].sum()

    print("=" * 80)
    print("BUILD-WIDE STATISTICS")
    print("=" * 80)
    print(f"Files processed: {len(json_files):,}")
    print(f"Total events: {len(events_df):,}")
    print(f"Total build time: {total_build_time_us / 1e6 / 60:.2f} minutes")
    print(f"Unique templates: {len(templates_df):,}")
    print(f"Template instantiations: {len(instantiations_df):,}")
    print(
        f"Template time: {total_template_time_us / 1e6 / 60:.2f} minutes ({100 * total_template_time_us / total_build_time_us:.1f}%)"
    )
    print("=" * 80)

### Top Templates by Total Time

Aggregate instantiations first, then join with templates for optimal performance.

⚠️ **Warning** Many of these times are identical. The join may be incorrect.

In [ ]:
if json_files and len(instantiations_df) > 0:
    # OPTIMIZATION: Aggregate FIRST, then join (much faster!)
    # This reduces 27M rows to ~20M unique templates before joining
    print("Aggregating template statistics...")
    start = time.time()

    template_stats = (
        instantiations_df.groupby("template_id")
        .agg({"dur_us": ["count", "sum", "mean", "median", "max"]})
        .reset_index()
    )

    template_stats.columns = [
        "template_id",
        "count",
        "total_us",
        "mean_us",
        "median_us",
        "max_us",
    ]

    # Now join with templates (much smaller aggregated dataset)
    template_stats = template_stats.merge(
        templates_df[
            ["template_id", "template_name", "full_signature", "depth", "arg_count"]
        ],
        on="template_id",
    )

    # Add computed columns
    template_stats["total_s"] = template_stats["total_us"] / 1e6
    template_stats["mean_ms"] = template_stats["mean_us"] / 1e3
    template_stats["median_ms"] = template_stats["median_us"] / 1e3
    template_stats["pct_template_time"] = (
        100 * template_stats["total_us"] / total_template_time_us
    )

    elapsed = time.time() - start
    print(f"Completed in {elapsed:.2f}s\n")

    print("Top 20 Templates by Total Time:")
    display(
        template_stats.nlargest(20, "total_us")[
            [
                "full_signature",
                "count",
                "total_s",
                "mean_ms",
                "median_ms",
                "depth",
                "pct_template_time",
            ]
        ]
    )

### Filter to CK Namespaces Only

Filter template statistics to show only `ck::` and `ck_tile::` namespaces, excluding standard library templates.

In [ ]:
if json_files and len(template_stats) > 0:
    # Filter to only CK namespaces
    ck_template_stats = template_stats[
        template_stats["template_name"].str.startswith("ck::", na=False)
        | template_stats["template_name"].str.startswith("ck_tile::", na=False)
    ].copy()

    # Recalculate percentage based on filtered CK templates only
    ck_total_time_us = ck_template_stats["total_us"].sum()
    ck_template_stats["pct_ck_time"] = (
        100 * ck_template_stats["total_us"] / ck_total_time_us
    )

    print(
        f"Filtered to {len(ck_template_stats):,} CK templates (from {len(template_stats):,} total)"
    )
    print(f"CK template time: {ck_total_time_us / 1e6:.2f}s")
    print(
        f"Percentage of total template time: {100 * ck_total_time_us / total_template_time_us:.1f}%"
    )

    print("\nTop 20 CK Templates by Total Time:")
    display(
        ck_template_stats.nlargest(20, "total_us")[
            [
                "full_signature",
                "count",
                "total_s",
                "mean_ms",
                "median_ms",
                "depth",
                "pct_ck_time",
            ]
        ]
    )

### Most Frequently Instantiated Templates

In [ ]:
if json_files and len(template_stats) > 0:
    print("Top 20 Most Frequently Instantiated Templates:")
    display(
        template_stats.nlargest(20, "count")[
            ["full_signature", "count", "total_s", "mean_ms", "depth"]
        ]
    )

## Part 3: Advanced Analysis

### Optimization Priority Score

Templates that are both frequently instantiated AND expensive per instantiation are prime optimization targets.

⚠️ **Warning** Many of these times are identical, `template_stats` may be incorrct.


In [ ]:
if json_files and len(template_stats) > 0:
    # Normalize count and mean to 0-1 range
    template_stats["count_norm"] = (
        template_stats["count"] - template_stats["count"].min()
    ) / (template_stats["count"].max() - template_stats["count"].min())
    template_stats["mean_norm"] = (
        template_stats["mean_us"] - template_stats["mean_us"].min()
    ) / (template_stats["mean_us"].max() - template_stats["mean_us"].min())

    # Priority score: weighted combination
    template_stats["priority_score"] = (
        0.5 * template_stats["count_norm"] + 0.5 * template_stats["mean_norm"]
    )

    print("Top 15 Optimization Targets (High Frequency + High Cost):")
    display(
        template_stats.nlargest(15, "priority_score")[
            ["full_signature", "count", "total_s", "mean_ms", "priority_score"]
        ]
    )

### Template Depth Distribution

Analyze how template nesting affects compilation time.

In [ ]:
if json_files and len(instantiations_df) > 0:
    # OPTIMIZATION: Aggregate by depth directly from instantiations, then join
    depth_stats = (
        instantiations_df.groupby("template_id")["dur_us"]
        .agg(["count", "sum", "mean", "median"])
        .reset_index()
    )

    # Join with templates to get depth
    depth_stats = depth_stats.merge(
        templates_df[["template_id", "depth"]], on="template_id"
    )

    # Now aggregate by depth
    depth_summary = (
        depth_stats.groupby("depth")
        .agg({"count": "sum", "sum": "sum", "mean": "mean", "median": "median"})
        .reset_index()
    )

    depth_summary.columns = ["depth", "count", "total_us", "mean_us", "median_us"]
    depth_summary["total_s"] = depth_summary["total_us"] / 1e6
    depth_summary["mean_ms"] = depth_summary["mean_us"] / 1e3
    depth_summary["median_ms"] = depth_summary["median_us"] / 1e3
    depth_summary["pct_total"] = (
        100 * depth_summary["total_us"] / total_template_time_us
    )

    print("Template Instantiation by Nesting Depth:")
    display(
        depth_summary[
            ["depth", "count", "total_s", "mean_ms", "median_ms", "pct_total"]
        ]
    )

### Template Argument Analysis

Analyze template argument patterns to understand dependencies.


⚠️ **Warning** All the arg counts are 8192, join may be incorrect.

In [ ]:
if json_files and len(template_args_df) > 0:
    # Count argument types
    arg_type_counts = template_args_df["arg_type"].value_counts()

    print("Template Argument Type Distribution:")
    print(f"{'Type':<15} {'Count':>10} {'Percentage':>12}")
    print("-" * 40)
    for arg_type, count in arg_type_counts.items():
        pct = 100 * count / len(template_args_df)
        print(f"{arg_type:<15} {count:>10,} {pct:>11.1f}%")

    # Templates with most arguments
    print("\nTemplates with Most Arguments:")
    arg_counts = (
        template_args_df.groupby("parent_template_id")
        .size()
        .reset_index(name="arg_count")
    )
    arg_counts = arg_counts.merge(
        templates_df[["template_id", "full_signature"]],
        left_on="parent_template_id",
        right_on="template_id",
    )
    display(arg_counts.nlargest(10, "arg_count")[["full_signature", "arg_count"]])

### Event Type Breakdown

In [ ]:
if json_files and len(events_df) > 0:
    event_stats = (
        events_df.groupby("name", observed=True)
        .agg({"dur": ["count", "sum", "mean", "max"]})
        .reset_index()
    )
    event_stats.columns = ["event_type", "count", "total_us", "mean_us", "max_us"]
    event_stats["total_min"] = event_stats["total_us"] / 1e6 / 60
    event_stats["mean_ms"] = event_stats["mean_us"] / 1e3
    event_stats["pct_total"] = 100 * event_stats["total_us"] / total_build_time_us

    print("Top 20 Event Types by Total Duration:")
    display(
        event_stats.nlargest(20, "total_us")[
            ["event_type", "count", "total_min", "mean_ms", "pct_total"]
        ]
    )

## Part 4: Build Parallelism Analysis

Analyze build-level parallelism using ninja's `.ninja_log` file to understand how well the build system utilizes available CPU cores.

### Parse Ninja Build Log


⚠️ **Warning** The `worker_id` values are all -1. We need to verify this data and sanity-check the timestamps. 


In [ ]:
from trace_analysis import NinjaLogParser

# Look for .ninja_log
ninja_log_paths = [
    Path("../../../build-trace/.ninja_log"),
]

ninja_log = None
for path in ninja_log_paths:
    if path.exists():
        ninja_log = path
        break

if ninja_log:
    print(f"Found ninja log: {ninja_log}")

    # Parse ninja log
    start = time.time()
    builds = NinjaLogParser.parse(ninja_log)
    elapsed = time.time() - start

    print(f"Parsed {len(builds):,} build events in {elapsed:.3f}s")

    # Convert to DataFrame
    builds_df = NinjaLogParser.to_dataframe(builds)
    print(f"\nBuilds DataFrame: {len(builds_df):,} rows")
    display(builds_df.head())
else:
    print("No .ninja_log found. Run a ninja build first.")

## Conclusion

This notebook demonstrated how to:

1. **Parse** Clang `-ftime-trace` files efficiently using parallel processing
2. **Transform** raw JSON into structured multi-table schemas
3. **Analyze** build performance using relational queries
4. **Visualize** build parallelism with Gantt charts
5. **Identify** optimization opportunities with data-driven techniques

### Key Advantages of Enhanced Schema

- **Memory efficient**: Normalized tables reduce redundancy
- **Query performance**: Optimized dtypes and indexes
- **Relational analysis**: Join tables to explore dependencies
- **Scalable**: Handles large builds with minimal memory

### Performance Optimizations

- **Aggregate first, join later**: Reduces dataset size before expensive joins
- **Optimized dtypes**: int32, int8, category types minimize memory
- **Parallel processing**: Utilizes all CPU cores for parsing
- **Efficient queries**: Pandas operations optimized for large datasets

### Build Parallelism Insights

TODO: See if we can recreate Perfetto view of the full ninja build.

### Next Steps

- Customize queries for your specific analysis needs
- Track build time trends over time
- Correlate template complexity with build times
- Optimize critical path builds
- Share insights with your team

### Resources

- [Clang -ftime-trace Documentation](https://releases.llvm.org/11.0.0/tools/clang/docs/ClangCommandLineReference.html#cmdoption-clang-ftime-trace)
- [Chrome Trace Event Format](https://docs.google.com/document/d/1CvAClvFfyA5R-PhYUmn5OOQtYMH4h6I0nSsKchNAySU/preview)
- [Ninja Build System](https://ninja-build.org/)
- [trace_analysis Library Documentation](../trace_analysis/README.md)